# Phase 7 — Figure Extraction & Vision-Language Reasoning

**Goal:** Make the assistant multimodal. We extract figures from the PDF, embed them with CLIP for cross-modal retrieval, and explain them with a local vision-language model (LLaVA via Ollama, or Qwen2-VL via Hugging Face).

This is the project's **differentiator** — "another RAG chatbot" becomes "a research assistant that can reason about diagrams."

By the end you will have:

1. A directory of extracted figure PNGs with `{doc_id, page, figure_index}` metadata.
2. CLIP embeddings for each figure stored alongside the text vectors.
3. A `vlm_explain_figure(image_path, surrounding_text)` function.
4. End-to-end: "explain the figure on page 3" returns a grounded multimodal explanation.


## 7.1 — Why CLIP + a VLM (and not just a VLM)?

A VLM can describe an image, but it cannot *find the right image* in a 50-PDF library. Splitting the job:

| Step                  | Model              | What it is good at                                |
|-----------------------|--------------------|----------------------------------------------------|
| Find relevant figure  | CLIP (text+image)  | Cheap, ranks images by text similarity.            |
| Explain that figure   | LLaVA / Qwen2-VL   | Grounded multimodal reasoning, slower per call.    |

This is the same retrieve-then-reason split that makes text RAG work — just applied to images.


## 7.2 — Extract figures with bounding-box-aware logic

PyMuPDF's `page.get_images()` only gets *embedded raster* images. Many papers have *vector* figures that are not single image streams. The robust approach:

1. Try embedded raster images first.
2. If a page has detected figure regions (via `page.get_text("blocks")` heuristics, or layout models like `LayoutLM`), rasterize just that region.
3. Save with metadata `{doc_id, page, figure_index, bbox}`.


In [ ]:
import os, sys, json, fitz, hashlib
from pathlib import Path
from pydantic import BaseModel
from typing import Optional

repo_root = Path.cwd()
while not (repo_root / "app").is_dir() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
os.chdir(repo_root)
sys.path.insert(0, str(repo_root))

PDF_PATH = Path("data/raw/attention_is_all_you_need.pdf")
FIG_DIR = Path("data/processed/figures"); FIG_DIR.mkdir(parents=True, exist_ok=True)

class Figure(BaseModel):
    figure_id: str
    doc_id: str
    source: str
    page: int
    image_path: str
    bbox: Optional[tuple[float, float, float, float]] = None
    caption: Optional[str] = None

def doc_id_of(path: Path) -> str:
    return f"{path.stem}_{hashlib.sha1(path.read_bytes()).hexdigest()[:10]}"

def extract_figures(pdf_path: Path) -> list[Figure]:
    doc = fitz.open(pdf_path)
    doc_id = doc_id_of(pdf_path)
    figs: list[Figure] = []
    for page_num, page in enumerate(doc, start=1):
        # Strategy 1: embedded raster images
        for idx, img in enumerate(page.get_images(full=True), start=1):
            xref = img[0]
            pix = fitz.Pixmap(doc, xref)
            if pix.n - pix.alpha > 3:
                pix = fitz.Pixmap(fitz.csRGB, pix)
            out = FIG_DIR / f"{doc_id}_p{page_num}_fig{idx}.png"
            out.write_bytes(pix.tobytes("png"))
            figs.append(Figure(
                figure_id=f"{doc_id}_p{page_num}_fig{idx}",
                doc_id=doc_id, source=pdf_path.name, page=page_num,
                image_path=str(out),
            ))
            pix = None
    return figs

figures = extract_figures(PDF_PATH)
print(f"Extracted {len(figures)} figures from {PDF_PATH.name}")
for f in figures[:5]:
    print(" -", f.figure_id, f.image_path)


## 7.3 — Inspect a figure


In [ ]:
from IPython.display import Image as IPyImage, display
if figures:
    display(IPyImage(filename=figures[0].image_path))


## 7.4 — CLIP embeddings for cross-modal retrieval

`open-clip-torch` ships a CPU-friendly `ViT-B/32` we can use for both text and image embeddings into the same space.


In [ ]:
import open_clip, torch
from PIL import Image
import numpy as np

model, _, preprocess = open_clip.create_model_and_transforms("ViT-B-32", pretrained="openai")
tokenizer = open_clip.get_tokenizer("ViT-B-32")
model.eval()

@torch.no_grad()
def clip_image_embedding(path: str) -> np.ndarray:
    img = preprocess(Image.open(path).convert("RGB")).unsqueeze(0)
    feat = model.encode_image(img)
    feat = feat / feat.norm(dim=-1, keepdim=True)
    return feat[0].cpu().numpy().astype("float32")

@torch.no_grad()
def clip_text_embedding(text: str) -> np.ndarray:
    tokens = tokenizer([text])
    feat = model.encode_text(tokens)
    feat = feat / feat.norm(dim=-1, keepdim=True)
    return feat[0].cpu().numpy().astype("float32")

# Embed every figure.
fig_embs = np.stack([clip_image_embedding(f.image_path) for f in figures]) if figures else np.zeros((0, 512))
print(f"CLIP image embeddings: {fig_embs.shape}")


## 7.5 — Cross-modal retrieval: text query -> best figure


In [ ]:
def find_best_figure(query: str, figures, embs) -> dict | None:
    if not figures:
        return None
    q = clip_text_embedding(query)
    scores = embs @ q
    i = int(np.argmax(scores))
    return {"figure": figures[i], "score": float(scores[i]), "rank": i}

for q in [
    "encoder decoder transformer architecture diagram",
    "scaled dot-product attention",
    "BLEU score table",
]:
    best = find_best_figure(q, figures, fig_embs)
    if best:
        print(f"'{q}' -> page {best['figure'].page}  score={best['score']:.3f}")
        print(f"   {best['figure'].image_path}")


## 7.6 — Explain the figure with a VLM

We default to **Ollama's `llava:7b`** because it runs locally with one `ollama pull` and no Hugging Face login. Below we also show the HF Transformers fallback with Qwen2-VL.


In [ ]:
import ollama, base64

def vlm_explain_figure_ollama(image_path: str, surrounding_text: str, model: str = "llava:7b") -> str:
    img_b64 = base64.b64encode(Path(image_path).read_bytes()).decode("utf-8")
    prompt = (
        "You are an AI teaching assistant. Explain this figure for a graduate student. "
        "Use the surrounding paper context to be precise. "
        "Be concrete about what the figure shows; avoid unsupported claims.\n\n"
        f"--- SURROUNDING TEXT ---\n{surrounding_text}\n--- END ---"
    )
    resp = ollama.chat(
        model=model,
        messages=[{"role": "user", "content": prompt, "images": [img_b64]}],
        options={"temperature": 0.2},
    )
    return resp["message"]["content"]

# Use the architecture diagram figure (page 3 in AIAYN is the model architecture).
target = find_best_figure("transformer model architecture encoder decoder", figures, fig_embs)
if target:
    fig = target["figure"]
    # Grab nearby text from page +/- 1
    doc = fitz.open(PDF_PATH)
    surround = "\n".join(doc[p-1].get_text() for p in [fig.page-1, fig.page, fig.page+1] if 1 <= p <= doc.page_count)[:2500]
    print(f"Explaining figure on page {fig.page} ({fig.image_path})\n")
    print(vlm_explain_figure_ollama(fig.image_path, surround))


## 7.7 — Hugging Face fallback (Qwen2-VL-2B)

If you do not want Ollama (or want the fully open-source HF stack), Qwen2-VL-2B-Instruct runs on a modern laptop GPU (8 GB) or, slowly, on CPU.


In [ ]:
# This is illustrative — uncomment only if you have GPU/RAM headroom.
# from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
# from PIL import Image
# model_id = "Qwen/Qwen2-VL-2B-Instruct"
# processor = AutoProcessor.from_pretrained(model_id)
# model_hf = Qwen2VLForConditionalGeneration.from_pretrained(model_id, torch_dtype="auto", device_map="auto")
#
# def vlm_explain_figure_hf(image_path: str, surrounding_text: str) -> str:
#     image = Image.open(image_path).convert("RGB")
#     messages = [{"role": "user", "content": [
#         {"type": "image", "image": image},
#         {"type": "text", "text": f"Explain this figure for a graduate student. Surrounding paper context:\n{surrounding_text}"},
#     ]}]
#     text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
#     inputs = processor(text=[text], images=[image], return_tensors="pt").to(model_hf.device)
#     out = model_hf.generate(**inputs, max_new_tokens=400)
#     return processor.batch_decode(out, skip_special_tokens=True)[0]
print("Uncomment the cell above to try the HF fallback.")


## 7.8 — Wire it into the API

In `app/api/main.py`, add:

```python
class ExplainRequest(BaseModel):
    query: str          # natural-language description of the figure
    doc_id: str | None = None

class ExplainResponse(BaseModel):
    figure_id: str
    page: int
    explanation: str

@app.post("/explain_figure", response_model=ExplainResponse)
def explain_figure(req: ExplainRequest, store: VectorStore = Depends(get_store)):
    fig, page_text = find_figure_and_context(req.query, doc_id=req.doc_id)
    return ExplainResponse(
        figure_id=fig.figure_id,
        page=fig.page,
        explanation=vlm_explain_figure_ollama(fig.image_path, page_text),
    )
```

And in `frontend/streamlit_app.py`, when the user picks "Explain figure" mode, call `/explain_figure` and render the returned image + explanation side by side.


## 7.9 — Caveats and honest limitations

- **Vector graphics.** Many newer papers (especially from Google/DeepMind) use vector PDFs where figures are *drawn* rather than embedded as raster images. PyMuPDF's `get_images` will miss them. Solution: layout-aware page rendering (`page.get_pixmap(clip=bbox, dpi=200)`) using bounding boxes from a layout model like `LayoutLMv3`.
- **VLM hallucination.** LLaVA-7B is small enough to mis-read complex diagrams. Always include surrounding text as grounding and treat VLM output as a hypothesis to verify, not a fact.
- **Latency.** A VLM call on CPU can take 20-60 s. Cache aggressively by `figure_id`.


## What you learned

- How to extract embedded figures from a PDF with metadata.
- How CLIP gives you a shared text-image space for cross-modal retrieval.
- A local VLM call (LLaVA via Ollama) to explain a figure with surrounding text.
- How to wire `POST /explain_figure` into the FastAPI app.
- The honest limits of multimodal retrieval and where to go next.

## Exercises

1. Add layout-aware figure extraction with `pdfplumber`'s `page.images` + `page.crop()`. Compare yield against PyMuPDF on a recent DeepMind paper.
2. Use CLIP scores to filter out non-figure images (paper headers, logos) — figures usually have score > 0.2 against "scientific figure" as a query.
3. Cache VLM outputs by `figure_id` in `data/processed/figure_captions.jsonl` so reruns are instant.

## Next: [Phase 8 — Teaching modes & prompts](./2026-05-25-phase08-teaching-modes-and-prompts.ipynb)
